# 1 · Talking to an LLM

⏱ about 15 minutes

In this workshop you build a chatbot that answers questions about three scientific papers
and shows you the page each answer comes from. Five short notebooks, one piece each:

1. **Chat** — talk to a language model *(this notebook)*
2. **Documents** — read PDFs and cut them into chunks
3. **Search** — find the chunks that match a question
4. **RAG** — answer from those chunks, with citations
5. **Together** — all pieces in one chatbot

![The two halves of RAG: preparing the documents once, and answering every question](img/overview.svg)

**How the notebooks drive the chatbot.** Some cells start with `%%writefile app.py`. They don't
run code; they *write* the chatbot's source file. A Chainlit server watches that file and
reloads whenever it changes. Run the cell, switch to the browser tab, and the bot has changed.

![A notebook cell writes app.py, Chainlit notices and reloads the browser](img/live-reload.svg)

Each notebook ends with **Your turn**: small changes to the chatbot that you try on your own.
Run the cells from top to bottom with **Shift+Enter**.

## Connect to the model

This reads your gateway address and key from `.env` and asks for one word.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]
llm = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
print(reply.choices[0].message.content)

Hello


An LLM call is a list of messages in, one message out. Let's ask about a detail from one of our papers.

In [2]:
QUESTION = "In Kage et al. (2018), what were the lifetime-encoded beads loaded with, and which lifetimes did they have?"

reply = llm.chat.completions.create(model=CHAT_MODEL, messages=[{"role": "user", "content": QUESTION}])
print(reply.choices[0].message.content)

In Kage et al. (2018), the lifetime-encoded beads were loaded with **quantum dots (QDs)**.

The beads were loaded with three different types of quantum dots to create distinct fluorescence lifetimes, which allowed for multiplexed detection. The approximate lifetimes used were:

*   **Short lifetime:** $\approx 2\text{ ns}$
*   **Medium lifetime:** $\approx 10\text{ ns}$
*   **Long lifetime:** $\approx 30\text{ ns}$

These beads were used as "lifetime barcodes" to enable the simultaneous tracking and identification of multiple targets within a single sample using Fluorescence Lifetime Imaging Microscopy (FLIM).


It sounds confident. Is it right? The model has never seen the paper, so the answer is general
knowledge at best and invented at worst. Notebook 4 asks the same question again, with the paper.

## Your first Chainlit app

[Chainlit](https://docs.chainlit.io) turns Python functions into a chat window. The function
decorated with `@cl.on_message` is called with every message you type.

In [3]:
%%writefile app.py
import chainlit as cl


@cl.on_message
async def on_message(message: cl.Message):
    await cl.Message(content=f"You said: {message.content}").send()

Writing app.py


Now start the server. It keeps running in the background, also while you work through the
other notebooks, and reloads whenever `app.py` changes.

In [4]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

Chatbot: http://localhost:8000


Open **http://localhost:8000** and say something.

> If the bot misbehaves, look at the end of the log: `!tail -20 chainlit.log`

## A real chatbot

Two things change compared to the notebook call above:
- **History.** The model has no memory, so we send the whole conversation every time.
  `cl.user_session` keeps it separately for every browser tab.
- **Streaming.** With `stream=True` the answer arrives token by token, so you watch it being typed.
  The app uses `AsyncOpenAI`, the same client in a form Chainlit can run without waiting.

In [5]:
%%writefile app.py
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# TODO 1: give the bot a personality.
SYSTEM_PROMPT = "You are a helpful assistant. Keep your answers short."

# TODO 2: add starter buttons here.


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


@cl.on_message
async def on_message(message: cl.Message):
    # TODO 3: start a stopwatch here ...
    history = cl.user_session.get("history")
    history.append({"role": "user", "content": message.content})

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history, stream=True
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)
    # ... and show the time under the answer here.
    await answer.send()

    history.append({"role": "assistant", "content": answer.content})

Overwriting app.py


## Your turn

Edit the `%%writefile app.py` cell above, run it again, and try your change in the browser.

Chat with the bot first: does it remember what you said two messages ago?

1. **Personality.** Change `SYSTEM_PROMPT` so the bot answers like a pirate, or only in German,
   or only in haiku. Start a new chat (the pencil icon, top left) to pick up the new prompt.
2. **Starter buttons.** Chainlit can show clickable example questions in an empty chat. Add a
   function decorated with `@cl.set_starters` that returns a list of
   `cl.Starter(label="…", message="…")`.
3. **Stopwatch.** Show under every answer how long it took. `time.perf_counter()` returns the
   current time in seconds; you can add text to `answer.content` before `answer.send()`.

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1** — any instruction works:
```python
SYSTEM_PROMPT = "You are a pirate. Answer every question like a pirate, in two sentences at most."
```

**2**
```python
@cl.set_starters
async def set_starters():
    return [
        cl.Starter(label="What is RAG?", message="Explain retrieval-augmented generation in two sentences."),
        cl.Starter(label="Tell a joke", message="Tell me a joke about scientists."),
    ]
```

**3** — `import time` at the top, then in `on_message`:
```python
    start = time.perf_counter()
    ...
    answer.content += f"\n\n*{time.perf_counter() - start:.1f} seconds*"
    await answer.send()
```
</details>